# Classificação de Grãos de Trigo — Seeds Dataset

FIAP 1TIAOA — Fase 4 Cap 3 (IR ALÉM)

Objetivo: aplicar CRISP-DM e Scikit-learn para classificar variedades de trigo (Kama, Rosa e Canadian) usando medidas morfológicas dos grãos.

## 1. Business Understanding

Cooperativas agrícolas precisam classificar grãos com consistência para reduzir erro humano, acelerar triagem e apoiar decisões sobre armazenamento, venda e controle de qualidade. O problema foi modelado como classificação multiclasse.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

DATA_PATH = Path('data/seeds.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('..') / 'data' / 'seeds.csv'
df = pd.read_csv(DATA_PATH)
df.head()

## 2. Data Understanding

O dataset tem 210 amostras e 7 variáveis explicativas: área, perímetro, compactação, comprimento do kernel, largura do kernel, assimetria e comprimento do sulco. A variável alvo é `variety`, com três classes.

In [ ]:
df.shape, df.dtypes

In [ ]:
df.describe()

In [ ]:
df.isna().sum()

## 3. Visualizações de EDA

As visualizações abaixo ajudam a enxergar distribuição, outliers e separação entre variedades.

In [ ]:
features = ['area','perimeter','compactness','length_kernel','width_kernel','asymmetry','groove_length']
class_names = {1: 'Kama', 2: 'Rosa', 3: 'Canadian'}

for col in features:
    plt.figure(figsize=(6, 3))
    sns.histplot(data=df, x=col, hue='variety', kde=True, palette='Set2')
    plt.title(f'Distribuição de {col}')
    plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df[features + ['variety']].corr(), annot=True, cmap='Blues', fmt='.2f')
plt.title('Matriz de correlação')
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df, x='area', y='perimeter', hue='variety', palette='Set2')
plt.title('Área x Perímetro por variedade')
plt.show()

## 4. Data Preparation

Foi usado split estratificado 70/30. Os modelos foram avaliados dentro de `Pipeline` com `StandardScaler`, mantendo o mesmo pré-processamento para comparação justa.

In [ ]:
X = df[features]
y = df['variety']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

X_train.shape, X_test.shape

## 5. Modeling

Foram comparados cinco classificadores: KNN, SVM, Random Forest, Naive Bayes e Regressão Logística. O enunciado exigia pelo menos três modelos; usamos cinco para ter comparação mais forte.

In [ ]:
candidates = {
    'knn': KNeighborsClassifier(),
    'svm': SVC(kernel='rbf'),
    'random_forest': RandomForestClassifier(random_state=42),
    'naive_bayes': GaussianNB(),
    'logistic_regression': LogisticRegression(max_iter=2000, random_state=42),
}

rows = []
fitted = {}
for name, estimator in candidates.items():
    pipe = Pipeline([('scaler', StandardScaler()), ('clf', estimator)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    fitted[name] = pipe
    rows.append({
        'model': name,
        'accuracy': accuracy_score(y_test, preds),
        'precision': precision_score(y_test, preds, average='weighted'),
        'recall': recall_score(y_test, preds, average='weighted'),
        'f1': f1_score(y_test, preds, average='weighted'),
    })

comparison = pd.DataFrame(rows).sort_values('f1', ascending=False)
comparison

## 6. Tuning

O melhor modelo inicial foi ajustado com GridSearchCV. No pipeline executado, o Random Forest ficou como melhor modelo.

In [ ]:
best_name = comparison.iloc[0]['model']
best_pipe = fitted[best_name]

param_grid = {'clf__n_estimators': [80, 120, 200]} if best_name == 'random_forest' else {}

if param_grid:
    grid = GridSearchCV(best_pipe, param_grid, cv=3)
    grid.fit(X_train, y_train)
    best_pipe = grid.best_estimator_
    print('Melhores parâmetros:', grid.best_params_)

preds = best_pipe.predict(X_test)
print(classification_report(y_test, preds, target_names=[class_names[i] for i in sorted(class_names)]))

In [ ]:
cm = confusion_matrix(y_test, preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names.values(), yticklabels=class_names.values())
plt.xlabel('Predito')
plt.ylabel('Real')
plt.title('Matriz de confusão')
plt.show()

## 7. Evaluation

Resultado registrado pelo pipeline:

- Melhor modelo: `random_forest`
- Acurácia: 0.9206
- Precisão ponderada: 0.9239
- Recall ponderado: 0.9206
- F1 ponderado: 0.9192

A matriz de confusão mostra desempenho muito forte em Canadian e Rosa, com maior dificuldade relativa na classe Kama.

## 8. Conclusão e próximos passos

O modelo é adequado como protótipo para triagem de grãos em cooperativas, mas ainda precisa de validação com dados reais antes de produção. Próximos passos: coletar novas safras, testar validação cruzada, criar interface de classificação e monitorar erros por lote.